## Locations

Paths are relative to the data directory given by the environment variable `M26_DATA_ROOT` (see the README); figure panels are written to `M26_RESULTS_ROOT`.

In [ ]:
# Locations (see README): M26_DATA_ROOT is required; M26_RESULTS_ROOT defaults to <repo>/results
import os
from pathlib import Path

REPO_ROOT = os.environ.get("M26_REPO_ROOT") or str(
    next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "environment" / "downstr.yml").is_file())
)
os.environ["M26_REPO_ROOT"] = REPO_ROOT
DATA_ROOT = os.environ["M26_DATA_ROOT"]
RESULTS_ROOT = os.environ.get("M26_RESULTS_ROOT", f"{REPO_ROOT}/results")
# Run in the directory this notebook originally ran in, so that relative paths resolve as they did
WORK_DIR = Path(DATA_ROOT, "notebooks/3UTR/organoids_knoblich/UMI")
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORK_DIR)

In [ ]:
import scanpy as sc
import pandas as pd

In [ ]:
BASEPATH = f"{DATA_ROOT}/3utr/organoids/knoblich/cellranger"

raw_counts_filename = f"{BASEPATH}/aggregated_annotated.h5ad"

# Load data


In [ ]:
adata_raw = sc.read_h5ad(raw_counts_filename)

display(adata_raw)

In [ ]:
adata_raw.var.gene_symbol.value_counts(dropna=False)

In [ ]:
adata_raw.var[adata_raw.var.gene_symbol == "None"]

In [ ]:
adata_raw.var[adata_raw.var.gene_symbol.isna()]

In [ ]:
adata_raw.var.head()

In [ ]:
adata_raw.obs.head()

In [ ]:
adata_raw.obs.celltype.value_counts(dropna=False)

In [ ]:
adata_raw = adata_raw[~adata_raw.obs.celltype.isna(), :]

In [ ]:
adata_raw.obs.celltype.value_counts(dropna=False)

In [ ]:
adata_raw.obs.induced.value_counts(dropna=False)

# Quality Control


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Annotate mitochondrial genes (human: MT- prefix in gene_symbol)
adata_raw.var["mt"] = adata_raw.var["gene_symbol"].str.startswith("MT-")
print(f"Number of MT genes: {adata_raw.var['mt'].sum()}")

# Calculate QC metrics from the count matrix
sc.pp.calculate_qc_metrics(
    adata_raw,
    qc_vars=["mt"],
    percent_top=None,
    log1p=False,
    inplace=True,
)

print(
    adata_raw.obs[["total_counts", "n_genes_by_counts", "pct_counts_mt"]]
    .describe()
    .round(2)
)

## QC Metrics — Overall Distribution


In [ ]:
sc.pl.violin(
    adata_raw,
    keys=["total_counts", "n_genes_by_counts", "pct_counts_mt"],
    jitter=0.4,
    multi_panel=True,
    size=0.5,
)

## QC Metrics — By Cell Type


In [ ]:
for groupby in ["celltype", "sample"]:
    sc.pl.violin(
        adata_raw,
        keys=["total_counts", "n_genes_by_counts", "pct_counts_mt"],
        groupby=groupby,
        jitter=0.3,
        rotation=30,
        size=0.3,
    )

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
sc_data = adata_raw.obs
scatter = ax.scatter(
    sc_data["total_counts"],
    sc_data["n_genes_by_counts"],
    c=sc_data["pct_counts_mt"],
    s=2,
    alpha=0.5,
    cmap="viridis",
)
ax.set_xlabel("Total counts (UMI)")
ax.set_ylabel("Genes detected")
ax.set_title("nUMI vs nGenes (color = % MT)")
plt.colorbar(scatter, ax=ax, label="% MT")

ax = axes[1]
ax.scatter(
    sc_data["total_counts"],
    sc_data["pct_counts_mt"],
    s=2,
    alpha=0.5,
    color="steelblue",
)
ax.set_xlabel("Total counts (UMI)")
ax.set_ylabel("% Mitochondrial counts")
ax.set_title("nUMI vs % MT")

plt.tight_layout()
plt.show()

## Filtering

The cells were pre-filtered upstream (min 998 genes, min 1118 UMI per cell)


In [ ]:
adata = adata_raw.copy()

# Filter genes: keep genes expressed in at least 3 cells
n_cells_before = adata.n_vars
sc.pp.filter_genes(adata, min_cells=3)
print(
    f"Genes: {n_cells_before} → {adata.n_vars}  ({n_cells_before - adata.n_vars} removed)"
)

In [ ]:
adata.var.head()

In [ ]:
# Set var index to gene symbols, then remove the source column
adata.var_names = adata.var["gene_symbol"].astype(str).values
adata.var_names_make_unique()

adata.var.drop(columns=["gene_symbol"], inplace=True)

adata.var.head()

## Normalization & Log-transformation


In [ ]:
# Store raw counts before normalization
adata.raw = adata.copy()

# Normalize to 10,000 counts per cell, then log1p-transform
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

print("Normalization done. X now holds log-normalized counts.")

## Highly Variable Genes


In [ ]:
sc.pp.highly_variable_genes(
    adata,
    n_top_genes=3000,
    flavor="seurat",
)
sc.pl.highly_variable_genes(adata)
print(f"HVGs selected: {adata.var['highly_variable'].sum()}")

## Dimensionality Reduction — PCA


In [ ]:
# Scale to unit variance (clip at 10); subset to HVGs for PCA
sc.pp.scale(adata, max_value=10)

sc.tl.pca(adata, svd_solver="arpack", use_highly_variable=True)

sc.pl.pca_variance_ratio(adata, n_pcs=50, log=True)

In [ ]:
# Color PCA by key metadata to get an early sense of structure
sc.pl.pca(
    adata,
    color=[
        "celltype",
        "sample",
        "nFeature_RNA",
        "n_genes_by_counts",
        "percent.mito",
        "pct_counts_mt",
    ],
    ncols=2,
    size=10,
    wspace=0.4,
)

## Neighborhood Graph & UMAP

> Choose `n_pcs` based on the elbow in the variance-ratio plot above.


In [ ]:
N_PCS = 30  # adjust after inspecting the elbow plot

sc.pp.neighbors(adata, n_neighbors=15, n_pcs=N_PCS)
sc.tl.umap(adata)

print("UMAP computed.")

## UMAP — Cell Type & Cluster Labels


In [ ]:
adata.obs.columns

In [ ]:
sc.pl.umap(
    adata,
    color=["celltype", "sample", "gRNA", "induced"],
    legend_loc="on data",
    legend_fontsize=8,
    ncols=2,
    size=10,
    title=["Subclass label", "Cluster label"],
)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Colors from scanpy default_20 palette
false_color = "#d62728"  # tab:red  (default_20 index 3)
true_color = "#1f77b4"  # tab:blue (default_20 index 0)
grey_color = "#D3D3D3"

# Temporary string-categorical column for reliable group matching
adata.obs["_induced_str"] = adata.obs["induced"].astype(str).astype("category")
# Categories alphabetically: False=0, True=1
codes = adata.obs["_induced_str"].cat.codes.values

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Plot 1: False highlighted (red), True greyed out
# Sort descending by code so False(0) is drawn last → on top
order1 = np.argsort(codes, kind="stable")[::-1]
sc.pl.umap(
    adata[order1],
    color="_induced_str",
    palette={"False": false_color, "True": grey_color},
    size=10,
    ax=axes[0],
    show=False,
    title="Non-induced  (induced = False)",
    legend_loc="on data",
    legend_fontsize=10,
)

# Plot 2: True highlighted (blue), False greyed out
# Sort ascending by code so True(1) is drawn last → on top
order2 = np.argsort(codes, kind="stable")
sc.pl.umap(
    adata[order2],
    color="_induced_str",
    palette={"False": grey_color, "True": true_color},
    size=10,
    ax=axes[1],
    show=False,
    title="Induced  (induced = True)",
    legend_loc="on data",
    legend_fontsize=10,
)

plt.tight_layout()
plt.show()

adata.obs.drop(columns=["_induced_str"], inplace=True)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter
from scipy.interpolate import RegularGridInterpolator


def grid_density(coords, n_bins=300, sigma=2):
    """
    Fast O(n) grid-based density estimate.
    Returns per-cell density values (same length as coords).
    """
    x, y = coords[:, 0], coords[:, 1]
    hist, xedges, yedges = np.histogram2d(x, y, bins=n_bins)
    hist_smooth = gaussian_filter(hist.T, sigma=sigma)  # shape (ny, nx)
    x_centers = (xedges[:-1] + xedges[1:]) / 2
    y_centers = (yedges[:-1] + yedges[1:]) / 2
    interp = RegularGridInterpolator(
        (y_centers, x_centers),
        hist_smooth,
        method="linear",
        bounds_error=False,
        fill_value=0.0,
    )
    return interp(np.column_stack([y, x]))


umap = adata.obsm["X_umap"]
mask_false = ~adata.obs["induced"].astype(bool)
mask_true = adata.obs["induced"].astype(bool)

density_false = grid_density(umap[mask_false])
density_true = grid_density(umap[mask_true])

# Clip at 99th percentile to avoid a few extreme cells washing out the scale
vmax_false = np.percentile(density_false, 99)
vmax_true = np.percentile(density_true, 99)

grey = "#D3D3D3"

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, mask, density, cmap, vmax, title in [
    (
        axes[0],
        mask_false,
        density_false,
        "Reds",
        vmax_false,
        "Non-induced (induced = False)",
    ),
    (axes[1], mask_true, density_true, "Blues", vmax_true, "Induced (induced = True)"),
]:
    # Background: the other group in grey — rasterized for speed
    ax.scatter(
        umap[~mask, 0],
        umap[~mask, 1],
        c=grey,
        s=3,
        alpha=0.2,
        linewidths=0,
        rasterized=True,
        zorder=1,
    )
    # Foreground: density-colored, drawn on top
    sc_plt = ax.scatter(
        umap[mask, 0],
        umap[mask, 1],
        c=density,
        cmap=cmap,
        vmin=0,
        vmax=vmax,
        s=3,
        linewidths=0,
        rasterized=True,
        zorder=2,
    )
    plt.colorbar(sc_plt, ax=ax, label="Density (a.u.)", shrink=0.8)
    ax.set_title(title, fontsize=12)
    ax.set_xlabel("UMAP 1")
    ax.set_ylabel("UMAP 2")
    ax.set_aspect("equal")
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
sc.pl.umap(
    adata,
    color=["total_counts", "n_genes_by_counts", "nFeature_RNA", "pct_counts_mt"],
    ncols=3,
    size=10,
    color_map="RdYlBu_r",
)

## Save Processed Object


In [ ]:
import os

out_dir = f"{BASEPATH}"
os.makedirs(out_dir, exist_ok=True)

out_path = f"{out_dir}/aggregated_annotated_filtered_normalized.h5ad"
adata.write_h5ad(out_path)
print(f"Saved: {out_path}")